In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [0]:
%skip
!unzip archive.zip

# Cargado e inspeccion de datos

In [0]:
%skip
customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')

order_items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

In [0]:
CATALOGO_ESQUEMA = "bronce" 

# 1. --- Carga de las Tablas como Spark DataFrames (DFs Distribuidos) ---
print(f"Cargando tablas desde el esquema '{CATALOGO_ESQUEMA}'...")

customers_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_customers_dataset")
orders_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_orders_dataset")
payments_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_order_payments_dataset")
order_items_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_order_items_dataset")
products_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_products_dataset")
category_translation_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.product_category_name_translation")
order_reviews_dataset_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_order_reviews_dataset")

print(f"Tablas cargadas a Spark DataFrames. Ejemplo: Registros de Clientes: {customers_spark_df.count()}")

# 2. --- Conversión a DataFrames de Pandas ---
# ¡ADVERTENCIA! Esta operación mueve todos los datos a la memoria de un solo nodo (Driver). 
# Solo debe usarse si el volumen total de datos es manejable (ej. < 1 GB).
print("\nConvirtiendo DataFrames de Spark a DataFrames de Pandas (.toPandas())...")

customers = customers_spark_df.toPandas()
orders = orders_spark_df.toPandas()
payments = payments_spark_df.toPandas()
order_items = order_items_spark_df.toPandas()
products = products_spark_df.toPandas()
category_translation = category_translation_spark_df.toPandas()
reviews= order_reviews_dataset_df.toPandas()

print(f"Conversión completada. Ejemplo: Tamaño del DataFrame de Clientes en Pandas: {len(customers)} filas.")
print(f"Primeros 5 registros del DataFrame de Clientes en Pandas:\n{customers.head()}")

# 3. --- Lista de DataFrames de Pandas para Procesamiento y Limpieza ---
# Ahora la lista 'datasets' contiene DataFrames de Pandas, lo que permite usar métodos nativos de Pandas.
datasets_dfs_pandas = [
    customers, 
    orders, 
    payments, 
    order_items, 
    products, 
    category_translation
]


## Customers

In [0]:
customers.head()

## Orders

In [0]:
orders.head()

## Payments

In [0]:
payments.head()

## Order - items

In [0]:
order_items.head()

## Productos

In [0]:
products.head()

## Category

In [0]:
category_translation.head()

In [0]:
plt.figure(figsize = (5, 7))

sns.stripplot(y = payments['payment_value']).set(xlabel = None, ylabel = None)
plt.title('Distribucion de los pagos', fontdict = {'fontsize': 12}, pad = 10.5)

In [0]:
plt.figure(figsize = (8, 5))

sns.countplot(x = 'payment_type', data = payments)
plt.title('Numero de ordenes por tipo de pago', fontdict = {'fontsize': 18}, pad = 10.5)
plt.xlabel('Payment type')
plt.ylabel('Count')

plt.tight_layout()

# Transformacion

In [0]:
# Filtramos solo pedidos entregados
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

In [0]:
# Convertimos la fecha clave a datetime
orders_delivered['order_purchase_timestamp'] = pd.to_datetime(orders_delivered['order_purchase_timestamp'])

# Tabla maestra

In [0]:
df_merged = customers.merge(orders_delivered, on='customer_id')

# 2. -> Payments
df_merged = df_merged.merge(payments, on='order_id')

# 3. -> Order Items
full_data = df_merged.merge(order_items, on='order_id')

# 4. -> Products
full_data = full_data.merge(products, on='product_id')

# 5. -> Category Translation
full_data = full_data.merge(category_translation, on='product_category_name')

full_data.head()

# Segmentacion y calculo de volumen

In [0]:
# Definimos "hoy" para los cálculos de Recencia y Antigüedad
hoy = full_data['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# Calculamos el volumen del producto (cm3)
full_data['product_volume_cm3'] = (
    full_data['product_length_cm'] * full_data['product_height_cm'] * full_data['product_width_cm']
)

# Generacion de RFM

In [0]:
rfm_logistica = full_data.groupby('customer_unique_id').agg(

    # Métricas RFM
    Recencia=('order_purchase_timestamp', lambda x: (hoy - x.max()).days),
    Frecuencia=('order_id', 'nunique'),
    Monetario=('payment_value', 'sum'),

    # Métricas de Antigüedad
    Antiguedad=('order_purchase_timestamp', lambda x: (hoy - x.min()).days),

    # Métricas de Comportamiento
    Amplitud_Categorias=('product_category_name_english', 'nunique'),
    Total_Articulos=('order_item_id', 'count'),

    # Métricas de Logística (Promedio)
    Avg_Peso_g=('product_weight_g', 'mean'),
    Avg_Volumen_cm3=('product_volume_cm3', 'mean')

).reset_index()

rfm_logistica.head(10)

In [0]:
rfm_logistica.tail(10)

## Escalado

In [0]:
# Definimos la lista de todas las features que alimentarán el modelo
features_list_final = [
    'Recencia',
    'Frecuencia',
    'Monetario',
    'Antiguedad',
    'Amplitud_Categorias',
    'Total_Articulos',
    'Avg_Peso_g',
    'Avg_Volumen_cm3'
]

# Separamos los datos a procesar
data_to_process_final = rfm_logistica[features_list_final].copy()

# Manejamos Nulos (ej. productos sin peso/volumen) rellenando con 0
data_to_process_final = data_to_process_final.fillna(0)

# 1. Transformación Logarítmica (para manejar outliers/sesgo)
data_log_final = np.log1p(data_to_process_final)

# 2. Escalado (para que todas las features tengan la misma importancia)
scaler_final = StandardScaler()
data_scaled_final = scaler_final.fit_transform(data_log_final)

print(data_scaled_final)

# Matriz de Correlacion

In [0]:
data_scaled_df = pd.DataFrame(data_scaled_final, columns=features_list_final)

# Calculamos la matriz de correlación
corr_matrix = data_scaled_df.corr()

# Graficamos el Heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # Ocultar la parte superior (espejo)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    mask=mask,
    vmin=-1,
    vmax=1
)
plt.title('Matriz de Correlación de Features (Técnica de Selección)', fontsize=16)
plt.show()

Se elimina la variable 'Antiguedad' por su alta correlacion con 'Recencia'

In [0]:
features_list_final = [
    'Recencia',
    'Frecuencia',
    'Monetario',
    # 'Antiguedad',
    'Amplitud_Categorias',
    'Total_Articulos',
    'Avg_Peso_g',
    'Avg_Volumen_cm3'
]

# Separamos los datos a procesar
data_to_process_final = rfm_logistica[features_list_final].copy()

# Manejamos Nulos (ej. productos sin peso/volumen) rellenando con 0
data_to_process_final = data_to_process_final.fillna(0)

# 1. Transformación Logarítmica (para manejar outliers/sesgo)
data_log_final = np.log1p(data_to_process_final)

# 2. Escalado (para que todas las features tengan la misma importancia)
scaler_final = StandardScaler()
data_scaled_final = scaler_final.fit_transform(data_log_final)

print(data_scaled_final)

## Analisis de Cohortes

In [0]:
# Usaremos df_merged para simplificar
df_cohorts = df_merged[['customer_unique_id', 'order_purchase_timestamp']].copy()

# Convertimos la fecha a un período mensual (ej. '2017-01')
df_cohorts['Order_Month'] = df_cohorts['order_purchase_timestamp'].dt.to_period('M')

print("--- Datos de Pedidos (Mes) ---")
print(df_cohorts.head())

In [0]:
# 1. Encontramos el primer mes de compra para CADA cliente
cohort_data = df_cohorts.groupby('customer_unique_id')['Order_Month'].min().reset_index()
cohort_data.columns = ['customer_unique_id', 'Cohort_Month']

# 2. Unimos esta cohorte de vuelta a todos sus pedidos
df_cohorts = df_cohorts.merge(cohort_data, on='customer_unique_id')

print("\n--- Pedidos con su Cohorte Asignada ---")
print(df_cohorts.head())

In [0]:
# 1. Calculamos la diferencia en meses (el offset)
def get_month_diff(x, y):
    # (x, y son períodos '2017-03', '2017-01')
    return (x.year - y.year) * 12 + (x.month - y.month)

df_cohorts['Month_Offset'] = df_cohorts.apply(
    lambda row: get_month_diff(row['Order_Month'], row['Cohort_Month']),
    axis=1
)

print("\n--- Pedidos con el Offset (Mes de Vida) ---")
print(df_cohorts.head())

In [0]:
# 1. Agrupamos por cohorte y offset
cohort_pivot = df_cohorts.groupby(['Cohort_Month', 'Month_Offset'])['customer_unique_id'].nunique().reset_index()

# 2. Creamos la tabla pivot
cohort_table = cohort_pivot.pivot_table(
    index='Cohort_Month',
    columns='Month_Offset',
    values='customer_unique_id'
)

# 3. Calculamos la retención (porcentaje)
#    Dividimos cada fila por el valor del 'Mes 0' (el tamaño inicial de la cohorte)
cohort_size = cohort_table.iloc[:, 0] # El tamaño de la cohorte es el Mes 0
retention_matrix = cohort_table.divide(cohort_size, axis=0) * 100

# 4. Limpiamos la matriz para el heatmap
retention_matrix = retention_matrix.round(2) # Redondeamos a 2 decimales

print("\n--- Matriz de Retención (%) ---")
print(retention_matrix)

In [0]:
plt.figure(figsize=(18, 10))
sns.heatmap(
    retention_matrix,
    annot=True,        # Muestra los números (porcentajes)
    fmt='.1f',         # Formato (1 decimal)
    cmap='viridis',    # Paleta de colores
    vmin=0,            # Valor mínimo (0%)
    vmax=5             # Valor máximo (ej. 5%) - ¡Ajusta esto!
)
plt.title('Heatmap de Retención de Clientes por Cohorte', fontsize=16)
plt.ylabel('Cohorte de Adquisición (Mes)')
plt.xlabel('Mes de Vida del Cliente (Offset)')
plt.show()

## Analisis de Monetario

In [0]:
plt.figure(figsize=(10, 8))

# 1. Creamos el Boxplot
sns.boxplot(
    y=rfm_logistica['Monetario'] # Ponemos Monetario en el eje Y
)

# 2. (Opcional) Limitamos el eje Y para que no sea tan alto
#    y podamos ver la 'caja'
# plt.ylim(0, rfm_logistica['Monetario'].quantile(0.95))

plt.title('Boxplot de Monetario - EL PROBLEMA', fontsize=16)
plt.ylabel('Gasto Total (R$)', fontsize=12)
plt.show()

In [0]:
plt.figure(figsize=(14, 7))

# 1. Creamos el gráfico con un color base neutro (ej. gris claro)
#    Guardamos el objeto del gráfico en la variable 'ax'
ax = sns.histplot(rfm_logistica['Recencia'], bins=50, kde=True, color='lightgrey')

plt.title('Distribución de Recencia (Coloreada por Segmentos)', fontsize=16)
plt.xlabel('Días desde la última compra', fontsize=12)
plt.ylabel('Conteo de Clientes', fontsize=12)

# 2. Iteramos sobre cada barra (patch) del histograma para cambiar su color
for p in ax.patches:
    # Obtenemos la posición X de la barra (el borde izquierdo) + la mitad del ancho
    # para saber en qué día cae el centro de la barra.
    x = p.get_x() + (p.get_width() / 2)

    # 3. Aplicamos la lógica de colores según tus rangos
    if 0 <= x <= 50:
        p.set_facecolor('blue')       # Nuevos (Segmento 4)
        p.set_alpha(0.7)              # Transparencia
        p.set_edgecolor('black')      # Borde para que se vea bien

    elif 250 <= x <= 300:
        p.set_facecolor('gold')       # La Gran Montaña de Churn
        p.set_alpha(0.8)
        p.set_edgecolor('black')

    elif 300 < x <= 600:
        p.set_facecolor('cyan')       # La Cola Larga
        p.set_alpha(0.7)
        p.set_edgecolor('black')

    else:
        # El resto se queda con el color por defecto (o puedes poner otro)
        p.set_facecolor('lightgrey')

# Añadimos una leyenda manual para explicar los colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', edgecolor='black', label='Recientes (0-50 días)'),
    Patch(facecolor='gold', edgecolor='black', label='Churn Masivo (250-300 días)'),
    Patch(facecolor='cyan', edgecolor='black', label='Cola Larga (300-600 días)'),
    Patch(facecolor='lightgrey', edgecolor='black', label='Otros rangos')
]
plt.legend(handles=legend_elements, title="Zonas de Interés")

plt.show()

En el grafico podemos distinguir 3 grupos (picos).

Grupo 1 (0-50): nuevos clientes

Grupo 2 (250-300): grupo de churn, clientes que hicieron compra unica.

Grupo 3 (300-600): cola de churn

## Analisis de la Frecuencia

In [0]:
# (Asumimos que 'rfm_logistica' ya está creado)

# 1. Mira la distribución de 'Frecuencia'
frecuencia_dist = rfm_logistica['Frecuencia'].value_counts(normalize=True).sort_index()

print("--- Distribución de Frecuencia (Porcentaje) ---")
print(frecuencia_dist.head())

# 2. Imprime el porcentaje de clientes con 1 sola compra
pct_una_compra = frecuencia_dist.loc[1] * 100
print(f"\nPorcentaje de clientes con 1 sola compra: {pct_una_compra:.2f}%")

In [0]:
plt.figure(figsize=(14, 7))
sns.countplot(
    x=rfm_logistica['Frecuencia'],
    palette='viridis'
)
plt.title('Distribución de Frecuencia (Escala Normal) - EL PROBLEMA', fontsize=16)
plt.xlabel('Número de Compras (Frecuencia)', fontsize=12)
plt.ylabel('Conteo de Clientes', fontsize=12)
plt.show()

In [0]:
print("\n--- Intentando aplicar pd.qcut (quintiles) a Frecuencia ---")

try:
    # Intentamos forzar la división en 5 grupos (quintiles)
    pd.qcut(rfm_logistica['Frecuencia'], 5, labels=[1, 2, 3, 4, 5])

except ValueError as e:
    print("\n¡FALLÓ COMO SE ESPERABA!")
    print(f"ERROR: {e}")

In [0]:
plt.figure(figsize=(14, 7))

# Usamos ax para poder modificar el eje
ax = sns.countplot(
    x=rfm_logistica['Frecuencia'],
    palette='viridis'
)

# ¡LA LÍNEA CLAVE!
ax.set_yscale('log') # Cambia el eje Y a escala logarítmica

plt.title('Distribución de Frecuencia (Escala Logarítmica) - LA SOLUCIÓN', fontsize=16)
plt.xlabel('Número de Compras (Frecuencia)', fontsize=12)
plt.ylabel('Conteo de Clientes (Escala Logarítmica)', fontsize=12)
plt.show()

"Como pd.qcut falla porque el 97% de los clientes tienen Frecuencia = 1, un método estadístico estándar no es aplicable.

Por lo tanto, debemos usar reglas de negocio manuales para crear segmentos de Frecuencia que sí tengan sentido para nosotros:

Puntuación 1: Clientes con 1 compra (el 97%).

Puntuación 3: Clientes con 2 compras (el 2-3%).

Puntuación 5: Clientes con 3+ compras (el <1%, nuestros verdaderos VIP)."

## Calculo RFM score

In [0]:
# 1. Creamos una nueva tabla solo para este análisis
rfm_scores = rfm_logistica[['customer_unique_id', 'Recencia', 'Frecuencia', 'Monetario']].copy()

# 2. Asignar R_Score (Puntuación de Recencia)
#    Usamos cuantiles (pd.qcut) porque la Recencia está bien distribuida.
#    Menor Recencia (más reciente) = Puntuación 5 (mejor)
rfm_scores['R_Score'] = pd.qcut(rfm_scores['Recencia'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# 3. Asignar F_Score (Puntuación de Frecuencia) - ¡MÉTODO MANUAL!
#    pd.qcut falla aquí. Usamos reglas de negocio.
#    Frecuencia > 2 = Puntuación 5 (Premium)
#    Frecuencia = 2 = Puntuación 3 (Recurrente)
#    Frecuencia = 1 = Puntuación 1 (Ocasional)
def score_frecuencia(f):
    if f > 2:
        return 5
    elif f == 2:
        return 3
    else:
        return 1

rfm_scores['F_Score'] = rfm_scores['Frecuencia'].apply(score_frecuencia)

# 4. Asignar M_Score (Puntuación Monetaria)
#    Usamos cuantiles. Más gasto = Puntuación 5 (mejor)
rfm_scores['M_Score'] = pd.qcut(rfm_scores['Monetario'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# 5. Crear el RFM Score (concatenado)
rfm_scores['RFM_Score'] = rfm_scores['R_Score'].astype(str) + \
                        rfm_scores['F_Score'].astype(str) + \
                        rfm_scores['M_Score'].astype(str)

print("\n--- Tabla con RFM Scores Clásicos ---")
rfm_scores.head()

### Segmentacion por reglas (RFM Score)

In [0]:
# 1. Definimos las condiciones (de mejor a peor)
#    Nota: Usamos los scores numéricos (R_Score, F_Score) para esto
conditions = [
    # Premium (R reciente, F alta, M alto)
    (rfm_scores['R_Score'] >= 4) & (rfm_scores['F_Score'] == 5) & (rfm_scores['M_Score'] >= 4),

    # Leal (R reciente, F recurrente)
    (rfm_scores['R_Score'] >= 4) & (rfm_scores['F_Score'] == 3),

    # Nuevo (R reciente, F baja)
    (rfm_scores['R_Score'] == 5) & (rfm_scores['F_Score'] == 1),

    # En Riesgo (R antigua, pero F/M eran altas)
    (rfm_scores['R_Score'] <= 2) & (rfm_scores['F_Score'] >= 3),

    # Perdido (R antigua, F baja)
    (rfm_scores['R_Score'] <= 2) & (rfm_scores['F_Score'] == 1)
]

# 2. Definimos los "bautizos"
choices = [
    'Premium (RFM)',
    'Leal (RFM)',
    'Nuevo (RFM)',
    'En Riesgo (RFM)',
    'Perdido (RFM)'
]

# 3. Aplicamos las reglas
rfm_scores['Segmento_RFM_Manual'] = np.select(conditions, choices, default='Otros')

print("\n--- Segmentación Manual por RFM Scores ---")
print(rfm_scores['Segmento_RFM_Manual'].value_counts())

## Modelo de Clustering Estadistico (K-Means)

### Metodo del codo

In [0]:
# 1. Calculamos la inercia para un rango de K
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k,
                    init='k-means++',  # Método de inicialización
                    n_init=10,         # Correr 10 veces con diferentes inicios
                    max_iter=300,
                    random_state=42)
    kmeans.fit(data_scaled_final)
    inertia.append(kmeans.inertia_)

# 2. Graficamos los resultados
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertia, 'bo-')
plt.xlabel('Número de Clústeres (K)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo (Elbow Method)')
plt.show()

In [0]:
#############################3
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import MiniBatchKMeans  # ⚡ Versión ultra-rápida
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# --- Preparación para ambos gráficos ---
K_range = range(2, 11)  # Rango de K (Silueta necesita al menos K=2)
inertia_list = []
silhouette_list = []

print("Calculando Inercia (Codo) y Silueta para K=2 a 10...")
print(f"Dataset: {data_scaled_final.shape[0]:,} registros")
print("-" * 50)

for k in K_range:
    print(f"Procesando K={k}...", end=" ", flush=True)
    
    # 1. Crear y entrenar el modelo MiniBatchKMeans (ultra-rápido)
    kmeans_model = MiniBatchKMeans(
        n_clusters=k,
        init='k-means++',
        n_init=3,              # ⚡ Reducido de 10 a 3
        max_iter=100,          # ⚡ Reducido de 300 a 100
        batch_size=1024,       # ⚡ Procesa en lotes de 1024
        random_state=42
    )
    kmeans_model.fit(data_scaled_final)
    
    # 2. Guardar la Inercia (para el Codo)
    inertia_list.append(kmeans_model.inertia_)
    
    # 3. Guardar el Índice de Silueta
    labels = kmeans_model.labels_
    
    # ⚡ Optimización: Si dataset > 20k, calcula Silueta con muestra
    if len(data_scaled_final) > 20000:
        sample_size = 10000
        sample_idx = np.random.choice(len(data_scaled_final), sample_size, replace=False)
        score = silhouette_score(data_scaled_final[sample_idx], labels[sample_idx])
    else:
        score = silhouette_score(data_scaled_final, labels)
    
    silhouette_list.append(score)
    print(f"✓ (Silueta: {score:.4f})")

print("-" * 50)
print("✅ Cálculos completados.\n")

# --- Creación de los Gráficos ---
plt.figure(figsize=(18, 7))

# Gráfico 1: Método del Codo (Inercia)
plt.subplot(1, 2, 1)
plt.plot(K_range, inertia_list, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clústeres (K)', fontsize=12)
plt.ylabel('Inercia (WCSS)', fontsize=12)
plt.title('Método del Codo (Buscando el "codo")', fontsize=14)
plt.grid(True, alpha=0.3)

# Gráfico 2: Índice de Silueta
plt.subplot(1, 2, 2)
plt.plot(K_range, silhouette_list, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Número de Clústeres (K)', fontsize=12)
plt.ylabel('Índice de Silueta', fontsize=12)
plt.title('Índice de Silueta (Buscando el "pico")', fontsize=14)
plt.grid(True, alpha=0.3)

# Marcar el mejor K
best_k_silhouette = K_range[np.argmax(silhouette_list)]
plt.axvline(x=best_k_silhouette, color='green', linestyle='--', linewidth=2,
            label=f'Mejor K={best_k_silhouette}')
plt.legend()

plt.tight_layout()
plt.show()

# Imprimir el mejor K según Silueta
print(f"\n⭐ El K con el Índice de Silueta más alto es: {best_k_silhouette}")
print(f"📊 Valor de Silueta: {max(silhouette_list):.4f}")

# --- Tabla resumen ---
print(f"\n{'K':<5} {'Inercia':<15} {'Silueta':<10}")
print("-" * 35)
for k, inercia, silueta in zip(K_range, inertia_list, silhouette_list):
    marker = "⭐" if k == best_k_silhouette else "  "
    print(f"{marker} {k:<3} {inercia:<15.2f} {silueta:<10.4f}")

## Entrenamiento del modelo con K

In [0]:
optimal_k_avanzado = 5
print(f"Entrenando modelo final con K={optimal_k_avanzado}...")

# Creamos y entrenamos el modelo final
kmeans_final_avanzado = KMeans(n_clusters=optimal_k_avanzado,
                               init='k-means++',
                               n_init=10,
                               max_iter=300,
                               random_state=42)

kmeans_final_avanzado.fit(data_scaled_final)

# Obtenemos las etiquetas (segmentos) para cada cliente
segmentos_avanzados = kmeans_final_avanzado.labels_

In [0]:
# Añadimos la columna 'Segmento' a la tabla correcta.
rfm_logistica['Segmento'] = segmentos_avanzados

# Analizamos el perfil usando la tabla 'rfm_logistica'
#    y la lista de features explícita para evitar errores.
segment_profile_avanzado = rfm_logistica.groupby('Segmento')[features_list_final].mean()
print("\n--- Perfil Promedio de Cada Segmento (Avanzado) ---")
print(segment_profile_avanzado)

## Comparacion RFM vs KMeans

In [0]:
# 1. (Recomendado) Bautizamos tus segmentos de K-Means para que sea fácil de leer
#    (Asegúrate de que los números coincidan con tu análisis anterior)
segment_kmeans_map = {
    2: 'Premium (K-Means)',
    4: 'Nuevo (K-Means)',
    1: 'Ballena Perdida (K-Means)',
    0: 'Perdido Voluminoso (K-Means)',
    3: 'Perdido Barato (K-Means)'
}
rfm_logistica['Segmento_KMeans_Nombre'] = rfm_logistica['Segmento'].map(segment_kmeans_map)

# 2. Preparamos la tabla de RFM Score para unirla
rfm_scores_map = rfm_scores[['customer_unique_id', 'Segmento_RFM_Manual', 'RFM_Score']]

# 3. Unimos ambas tablas de segmentación
df_comparacion = rfm_logistica.merge(rfm_scores_map, on='customer_unique_id')

# 4. ¡Creamos la Matriz de Confusión (Crosstab)!
print("\n" + "="*50)
print(" COMPARACIÓN: K-MEANS (Filas) vs. RFM SCORE (Columnas)")
print("="*50)

comparison_matrix = pd.crosstab(
    df_comparacion['Segmento_KMeans_Nombre'],
    df_comparacion['Segmento_RFM_Manual']
)

# 5. Mostramos la matriz de comparación
comparison_matrix

In [0]:
# (Asumimos que 'comparison_matrix' ya existe del paso anterior)

# 1. Normalizamos la matriz por FILA (por K-Means)
#    Esto convierte los conteos en porcentajes de 0.0 a 1.0
comparison_normalized = comparison_matrix.div(comparison_matrix.sum(axis=1), axis=0)

# 2. Multiplicamos por 100 para ver porcentajes
comparison_normalized = comparison_normalized * 100

print("\n--- Composición Porcentual de Segmentos K-Means ---")
print(comparison_normalized.to_string(formatters={None: '{:,.1f}%'.format}))

# 3. Creamos el gráfico de barras apiladas
ax = comparison_normalized.plot(
    kind='barh',    # Gráfico de barras horizontal (más fácil de leer)
    stacked=True,   # ¡La clave! Apila los valores
    figsize=(14, 8),
    cmap='viridis'  # Paleta de colores
)

plt.title('Composición de Segmentos K-Means (vistos por RFM Manual)', fontsize=16)
plt.xlabel('Porcentaje (%)')
plt.ylabel('Segmento K-Means (Avanzado)')
plt.legend(title='Segmento RFM Manual', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Agregamos categoria de productos a los segmentos

In [0]:
# 1. Creamos una tabla que mapea a cada cliente con su segmento
customer_segments_map = rfm_logistica[['customer_unique_id', 'Segmento']]

# 2. Tomamos los datos originales de compra (que tienen las categorías)
customer_purchases = full_data[['customer_unique_id', 'product_category_name_english']]

# 3. Unimos el segmento de cada cliente con CADA compra que hizo
segment_purchases = customer_purchases.merge(customer_segments_map, on='customer_unique_id')

# 4. Ahora, contamos cuántas veces aparece cada categoría DENTRO de cada segmento
category_by_segment = segment_purchases.groupby(
    ['Segmento', 'product_category_name_english']
).size().reset_index(name='count')

# 5. Ordenamos para ver las más populares
category_by_segment_sorted = category_by_segment.sort_values(
    by=['Segmento', 'count'], ascending=False
)

# 6. Mostramos las 4 categorías más compradas para cada segmento
top_categories = category_by_segment_sorted.groupby('Segmento').head(4)

top_categories

# Descripcion de los Segmentos

# Segmento "2": Premium Leal (El Cliente Ideal)

"Pocos, pero vitales. Los verdaderos fans de la marca."

Perfil: Son clientes que han superado la barrera de la compra única. No solo compran repetidamente (Frecuencia 1.85), sino que compran en múltiples departamentos (Amplitud > 2).

Datos Clave:

Frecuencia: Más Alta (Son los únicos recurrentes).

Amplitud: Más Alta (Generalistas: Hogar, Deportes, Tecnología).

Eficiencia: Generan 2.43x más ingresos que el cliente promedio.

Categoría Top: bed_bath_table, furniture_decor, sports_leisure.

Estrategia: Retención y Fidelización (VIP).

Acción: Crear un programa de lealtad. Enviar encuestas para entender qué mejorar (su review score era 3.9, ¡hay riesgo!). Ofrecer envíos gratis o acceso anticipado.


---



# Segmento "3": Ballena (El Comprador de Alto Valor)
"Una sola cita, pero inolvidable. El dinero rápido."

Perfil: Clientes que hicieron una inversión masiva en una sola transacción o en un periodo muy corto. Gastaron mucho dinero, pero no mostraron lealtad a largo plazo (aún).

Datos Clave:

Monetario: Más Alto (Ticket promedio ~R$ 670).

Eficiencia: 3.16x (Son las estrellas financieras).

Frecuencia: Baja (~1.0).

Categoría Top: computers_accessories, watches_gifts (Lujo/Tecnología).

Estrategia: Reactivación de Alto Valor.

Acción: No les ofrezcas descuentos pequeños. Ofréceles garantías extendidas, accesorios complementarios para lo que compraron (Cross-sell) o lanzamientos exclusivos. El objetivo es conseguir la segunda compra a toda costa.



---



# Segmento "1": Voluminoso (El Desafío Logístico)
"El peso pesado del negocio. Facturación alta, margen dudoso."

Perfil: Clientes que compran artículos grandes y pesados (muebles, electrodomésticos). Representan un tercio de tu base y de tus ingresos, pero son costosos de servir.

Datos Clave:

Peso/Volumen: Más Altos (Compras de >4kg en promedio).

Recencia: Alta (Suelen ser compras únicas de necesidad puntual).

Eficiencia: 1.09x (Promedio).

Categoría Top: furniture_decor, bed_bath_table (Muebles).

Estrategia: Optimización de Margen.

Acción: No gastar mucho presupuesto de marketing en reactivarlos (es difícil que alguien compre dos sofás seguidos). Enfocar esfuerzos en el equipo de Logística para reducir costos de envío y mejorar la experiencia de entrega (que suele ser lenta para este grupo).



---



# Segmento "0": Nuevo (La Cantera)
"Sangre nueva. El futuro del negocio se decide aquí."

Perfil: Clientes recién adquiridos. Su historia con la marca apenas comienza. Tienen el potencial de convertirse en Premium o irse al segmento Barato.

Datos Clave:

Recencia: Más Baja (~40 días).

Antigüedad: Mínima.

Categoría Top: health_beauty, bed_bath_table.

Estrategia: Activación y Onboarding.

Acción: Este es el momento crítico. Enviar campaña de bienvenida ("Welcome Series"). Incentivar la segunda compra inmediatamente con un descuento por tiempo limitado ("Tu cupón vence en 7 días").



---



# Segmento "4": Bajo costo (El Ruido)
"Muchos clientes, poco valor. La cola larga."

Perfil: El grupo más grande en cantidad de personas, pero el que menos aporta. Compraron algo pequeño, ligero y de bajo costo una vez y desaparecieron.

Datos Clave:

Monetario: Más Bajo (Ticket ~R$ 92).

Peso: Más bajo (Paquetes ligeros).

Eficiencia: 0.43x (Muy ineficiente).

Categoría Top: telephony (Cables, fundas), health_beauty (Cosméticos baratos).

Estrategia: Automatización o Ignorar.

Acción: No invertir presupuesto publicitario manual. Dejarlos en campañas de email marketing 100% automatizadas y de bajo costo. Si vuelven, bien; si no, no afectan la rentabilidad.

In [0]:
data_scaled_final_df = pd.DataFrame(data_scaled_final, columns=features_list_final)

# 2. Añadimos la columna Segmento
data_scaled_final_df['Segmento'] = segmentos_avanzados

# 3. Calculamos la media de los datos escalados por segmento
profile_scaled_avg = data_scaled_final_df.groupby('Segmento').mean()

# --- Ahora, creamos el Heatmap ---
plt.figure(figsize=(14, 6))
sns.heatmap(
    profile_scaled_avg,
    annot=True,     # Muestra los números en cada celda
    cmap='coolwarm', # Colores: azul (frío/bajo) a rojo (caliente/alto)
    fmt='.2f'       # Formatea los números a 2 decimales
)
plt.title('Perfil Promedio de Segmentos (Datos Escalados)', fontsize=16)
plt.xlabel('Métricas (Features)')
plt.ylabel('Segmento')
plt.show()

In [0]:
g = sns.catplot(
    data=top_categories,
    x='product_category_name_english',
    y='count',
    col='Segmento',       # Crea una columna separada para cada Segmento
    kind='bar',
    col_wrap=3,           # Cuántos gráficos poner por fila (ej. 3)
    sharex=False,         # Permite que las etiquetas del eje X varíen
    height=4,             # Altura de cada gráfico
    aspect=1.2            # Relación de aspecto
)

# Ajustes para que se lea bien
g.set_xticklabels(rotation=45, ha='right') # Rota las etiquetas de categorías
g.set_titles("Segmento {col_name}", size=14) # Títulos para cada gráfico
g.set_axis_labels("Categoría de Producto", "Conteo de Artículos")
plt.suptitle('Top 4 Categorías Más Compradas por Segmento', y=1.03, size=18)
plt.tight_layout()
plt.show()

In [0]:
# 1. Cargar la única tabla nueva que necesitamos
try:
    reviews = order_reviews_dataset_df.toPandas()#pd.read_csv('olist_order_reviews_dataset.csv')
    print("Tabla 'reviews' cargada.")
except FileNotFoundError:
    print("Error: No se encontró 'olist_order_reviews_dataset.csv'.")
    raise

# 2. Crear nuestro "mapa" de segmentos
# Contiene: customer_unique_id | Segmento
segment_map = rfm_logistica[['customer_unique_id', 'Segmento']]

# 3. Unir el 'Segmento' a nuestra tabla 'full_data'
# Ahora 'full_data' tiene Segmento, Categoría, Vendedor, Cliente, etc.
full_data_segmented = full_data.merge(segment_map, on='customer_unique_id')

# 4. Unir las reseñas a esta tabla maestra
# Usamos how='left' para no perder pedidos que quizás no tengan reseña
full_data_segmented = full_data_segmented.merge(reviews, on='order_id', how='left')

print("\n--- ¡Tabla Maestra de Análisis Creada! ---")
print(full_data_segmented.head())

In [0]:
print("\n--- Análisis 1: Geoespacial (Top 5 Estados por Segmento) ---")

# 1. Contamos clientes únicos por estado y segmento
geo_analysis = full_data_segmented.groupby(
    ['Segmento', 'customer_state']
)['customer_unique_id'].nunique().reset_index(name='customer_count')

# 2. Ordenamos para ver los más populares
geo_analysis_sorted = geo_analysis.sort_values(
    by=['Segmento', 'customer_count'], ascending=False
)

# 3. Mostramos el Top 5
top_5_geo = geo_analysis_sorted.groupby('Segmento').head(5)

print(top_5_geo)

In [0]:
# 1. Creamos la tabla pivot (matriz)
geo_pivot = geo_analysis.pivot_table(
    index='Segmento',
    columns='customer_state',
    values='customer_count'
)

# 2. (Opcional) Tomemos solo los 10 estados principales para que sea legible
top_10_states = geo_analysis.groupby('customer_state')['customer_count'].sum().nlargest(10).index
geo_pivot_top10 = geo_pivot[top_10_states]

# 3. Creamos el Heatmap
plt.figure(figsize=(16, 6))
sns.heatmap(
    geo_pivot_top10,
    annot=True,     # Muestra los números
    fmt='.0f',      # Números sin decimales
    cmap='viridis', # Paleta de colores
    linewidths=.5
)
plt.title('Concentración de Clientes por Segmento y Estado (Top 10 Estados)', fontsize=16)
plt.xlabel('Estado del Cliente')
plt.ylabel('Segmento K-Means')
plt.show()

In [0]:
print("\n--- Análisis 2 (Mejorado): Satisfacción vs. Volumen ---")

# 1. Calculamos la media Y el conteo de 'review_score'
satisfaction_analysis_adv = full_data_segmented.groupby('Segmento')['review_score'].agg(
    review_score_avg='mean',
    review_count='count'
).reset_index()

satisfaction_analysis_adv = satisfaction_analysis_adv.sort_values(by='review_score_avg', ascending=False)

print(satisfaction_analysis_adv)

In [0]:
print("\n--- Análisis 2 (Avanzado): Calidad vs. Tasa de Reseña ---")

# (Asumimos que 'full_data_segmented' y 'satisfaction_analysis_adv' ya existen)

# 1. Calculamos el total de PEDIDOS ÚNICOS por segmento
total_orders_seg = full_data_segmented.groupby('Segmento')['order_id'].nunique().reset_index(name='total_orders')

# 2. Recalculamos el total de RESEÑAS ÚNICAS por segmento
#    (Usamos 'review_id' para asegurarnos de que contamos reseñas, no items)
total_reviews_seg = full_data_segmented.dropna(subset=['review_id']).groupby('Segmento')['review_id'].nunique().reset_index(name='total_reviews')

# 3. Unimos estos conteos con nuestra tabla de promedios
satisfaction_final = satisfaction_analysis_adv.merge(total_orders_seg, on='Segmento')
satisfaction_final = satisfaction_final.merge(total_reviews_seg, on='Segmento')

# 4. ¡Calculamos la Tasa de Reseña!
#    (Total de Reseñas / Total de Pedidos) * 100
satisfaction_final['review_rate_pct'] = (satisfaction_final['total_reviews'] / satisfaction_final['total_orders']) * 100

# 5. Añadimos los nombres para la leyenda
segment_kmeans_map = {
    2: 'Premium Leal',
    4: 'Cliente Nuevo',
    1: 'Ballena Perdida',
    0: 'Perdido (Voluminoso)',
    3: 'Perdido (Bajo costo)'
}
satisfaction_final['Segmento_Nombre'] = satisfaction_final['Segmento'].map(segment_kmeans_map)

# 6. Mostramos la nueva tabla de análisis
print("\n--- Tabla de Análisis de Satisfacción Final ---")
print(satisfaction_final[['Segmento_Nombre', 'review_score_avg', 'review_rate_pct', 'total_orders']])


# 2. Graficamos los resultados
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=satisfaction_final,
    x='review_score_avg',    # Eje X: Calidad (Promedio 1-5)
    y='review_rate_pct',     # Eje Y: Tasa de Reseña (%)
    hue='Segmento_Nombre',   # Color por segmento
    s=200,
    palette='viridis'
)

plt.title('Calidad de Satisfacción vs. Tasa de Reseña por Segmento', size=16)
plt.xlabel('Review Score Promedio (Calidad)', size=12)
plt.ylabel('Tasa de Reseña (% de Pedidos con Reseña)', size=12)
plt.legend(title='Segmento')
plt.grid(True)
plt.show()

In [0]:
print("\n--- Análisis 3: Logística (Tiempos de Entrega) ---")

# 1. (Feature Engineering) Calculamos métricas de entrega
#    Necesitamos convertir las fechas (si no lo están ya en 'full_data_segmented')
full_data_segmented['order_delivered_customer_date'] = pd.to_datetime(full_data_segmented['order_delivered_customer_date'])
full_data_segmented['order_purchase_timestamp'] = pd.to_datetime(full_data_segmented['order_purchase_timestamp'])
full_data_segmented['order_estimated_delivery_date'] = pd.to_datetime(full_data_segmented['order_estimated_delivery_date'])

#    A. Tiempo total de entrega
full_data_segmented['tiempo_entrega_dias'] = (
    full_data_segmented['order_delivered_customer_date'] - full_data_segmented['order_purchase_timestamp']
).dt.days

#    B. Días de retraso (Negativo = llegó tarde, Positivo = llegó temprano)
full_data_segmented['entrega_vs_estimado_dias'] = (
    full_data_segmented['order_estimated_delivery_date'] - full_data_segmented['order_delivered_customer_date']
).dt.days

# 2. Calculamos el promedio de estas métricas por segmento
logistics_analysis = full_data_segmented.groupby('Segmento')[
    ['tiempo_entrega_dias', 'entrega_vs_estimado_dias']
].mean().reset_index()

print(logistics_analysis)

In [0]:
print("\n--- Análisis 4: Vendedores (Diversidad de Vendedores) ---")

# 1. Calculamos cuántos vendedores únicos ha usado cada cliente
customer_seller_diversity = full_data_segmented.groupby(
    'customer_unique_id'
)['seller_id'].nunique().reset_index(name='vendedores_unicos')

# 2. Unimos esto con nuestro mapa de segmentos
customer_seller_diversity = customer_seller_diversity.merge(segment_map, on='customer_unique_id')

# 3. Calculamos la diversidad de vendedores promedio por segmento
seller_analysis = customer_seller_diversity.groupby('Segmento')['vendedores_unicos'].mean().reset_index()

print(seller_analysis.sort_values(by='vendedores_unicos', ascending=False))

In [0]:
# Supongamos que ya ejecutaste el pipeline y tienes 'rfm_logistica' con la columna 'Segmento'

print("=== REPORTE AUTOMÁTICO DEL PIPELINE (MES ACTUAL) ===\n")

# 1. Cálculo de Métricas Técnicas
sil_score = silhouette_score(data_scaled_final, segmentos_avanzados)
print(f"1. SALUD TÉCNICA:")
print(f"   - Índice de Silueta: {sil_score:.4f}")
print(f"   - Estado: {'🟢 Óptimo' if sil_score > 0.25 else '🔴 Revisar'}\n")

# 2. Cálculo de Métricas de Negocio
print(f"2. SALUD DEL NEGOCIO (Distribución de Segmentos):")

# Agrupamos para obtener conteo y suma de dinero
resumen_negocio = rfm_logistica.groupby('Segmento').agg(
    Clientes=('customer_unique_id', 'count'),
    Ingresos=('Monetario', 'sum')
)

# Calculamos porcentajes
total_clientes = resumen_negocio['Clientes'].sum()
total_ingresos = resumen_negocio['Ingresos'].sum()

resumen_negocio['% Clientes'] = (resumen_negocio['Clientes'] / total_clientes) * 100
resumen_negocio['% Ingresos'] = (resumen_negocio['Ingresos'] / total_ingresos) * 100

# Mapeamos los nombres para que sea legible
mapa_nombres = {
    2: 'Premium',      # Frecuencia y Amplitud altas
    3: 'Ballena',      # Monetario más alto
    0: 'Nuevo',        # Recencia más baja
    1: 'Voluminoso',   # Peso más alto
    4: 'Bajo costo'        # Monetario y Peso más bajos
}
resumen_negocio.index = resumen_negocio.index.map(mapa_nombres)

print(resumen_negocio[['% Clientes', '% Ingresos']].sort_values(by='% Ingresos', ascending=False))

# Reporte Pipeline

In [0]:
print("Generando Reporte Detallado del Pipeline...\n")

# --- PASO 1: Calcular Métricas Financieras y Operativas ---
# Agrupamos por Segmento para obtener promedios y sumas
reporte_base = rfm_logistica.groupby('Segmento').agg(
    Conteo=('customer_unique_id', 'count'),
    Ingresos_Total=('Monetario', 'sum'),
    Ticket_Promedio=('Monetario', 'mean'),
    Recencia_Media=('Recencia', 'mean')
)

# Calculamos los porcentajes y el índice
total_clientes = reporte_base['Conteo'].sum()
total_ingresos = reporte_base['Ingresos_Total'].sum()

reporte_base['% Clientes'] = (reporte_base['Conteo'] / total_clientes) * 100
reporte_base['% Ingresos'] = (reporte_base['Ingresos_Total'] / total_ingresos) * 100
reporte_base['Indice_Valor'] = reporte_base['% Ingresos'] / reporte_base['% Clientes']

# --- PASO 2: Encontrar la Categoría Top por Segmento ---
# Necesitamos volver a 'full_data' para esto
df_categorias = full_data[['customer_unique_id', 'product_category_name_english']].merge(
    rfm_logistica[['customer_unique_id', 'Segmento']],
    on='customer_unique_id'
)

# Contamos categorías por segmento
top_cat_df = df_categorias.groupby(['Segmento', 'product_category_name_english']).size().reset_index(name='counts')
# Ordenamos y tomamos la primera (la más vendida) de cada segmento
top_cat_df = top_cat_df.sort_values(['Segmento', 'counts'], ascending=[True, False])
top_cat_df = top_cat_df.groupby('Segmento').head(1).set_index('Segmento')

# Añadimos la categoría al reporte base
reporte_base['Categoria_Top'] = top_cat_df['product_category_name_english']

# --- PASO 3: Formateo y "Bautizo" ---
# Aplicamos los nombres de tu diccionario
reporte_base.index = reporte_base.index.map(mapa_nombres)

# Ordenamos por Importancia (% Ingresos)
reporte_final = reporte_base.sort_values(by='% Ingresos', ascending=False)

# --- PASO 4: IMPRIMIR EL REPORTE VISUAL ---

print("="*80)
print("  REPORTE MENSUAL DE SEGMENTACIÓN (PIPELINE AUTOMATIZADO)")
print("="*80)
print(f"Total Clientes Analizados: {total_clientes:,}")
print(f"Total Ingresos Analizados: R$ {total_ingresos:,.2f}")
print("-" * 80)

# Encabezados
print(f"{'SEGMENTO':<20} | {'% CLIENTES':<12} | {'% INGRESOS':<12} | {'INDICE':<8} | {'GASTO PROMEDIO':<10} | {'RECENCIA':<10} | {'CATEGORIA TOP'}")
print("-" * 80)

# Filas
for segmento, row in reporte_final.iterrows():
    print(f"{segmento:<20} | "
          f"{row['% Clientes']:>10.1f}% | "
          f"{row['% Ingresos']:>10.1f}% | "
          f"{row['Indice_Valor']:>8.2f}x | "
          f"R$ {row['Ticket_Promedio']:>7.0f} | "
          f"{row['Recencia_Media']:>5.0f} dias | "
          f"{row['Categoria_Top']}")

print("="*80)

# --- PASO 5: GENERAR ACCIONES AUTOMÁTICAS ---
print("\nACCIONES RECOMENDADAS (Lógica Automática):")
for segmento, row in reporte_final.iterrows():
    if "Ballena" in segmento:
        print(f"> 💰 {segmento}: Activar campaña de 'Reactivación VIP' (Oferta Alto Valor). Objetivo: {int(row['Conteo'])} clientes.")
    elif "Premium" in segmento:
        print(f"> 💎 {segmento}: Enviar 'Novedades de Hogar' y encuesta de satisfacción. Objetivo: Retención.")
    elif "Nuevo" in segmento:
        print(f"> 👋 {segmento}: Activar flujo de 'Segunda Compra' (Cross-selling). Objetivo: Conversión.")
    elif "Voluminoso" in segmento:
        print(f"> 😴 {segmento}: Monitorear costos logísticos. No priorizar inversión en marketing.")

"Validación del Modelo: El Índice de Silueta resultante fue de 0.29.

Si bien en datasets sintéticos se buscan valores superiores a 0.5, en segmentación conductual de clientes (donde las fronteras entre comportamientos son difusas y continuas), un valor en el rango de 0.25 - 0.35 se considera un estándar de industria aceptable.

Este valor confirma que existe una estructura de grupos real (no es ruido aleatorio), pero que los grupos son adyacentes. Se priorizó la elección de K=5 sobre un K menor (que tendría mayor silueta) porque K=5 reveló perfiles de negocio críticos ('Ballenas' y 'Voluminosos') que quedaban ocultos en agrupaciones más simples."